In [1]:
    import pandas as pd
    import spacy
    import re
    import nltk
    from nltk.corpus import stopwords

    # 1. Gerekli kütüphaneleri ve modelleri yükle
    nltk.download('stopwords', quiet=True)

    # Not: en_core_web_sm ve de_core_news_sm isimleri sende farklıysa (md vb.) güncelleyebilirsin
    nlp_en = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    nlp_de = spacy.load("de_core_news_sm", disable=["parser", "ner"])
    nlp_tr = spacy.load("tr_core_news_lg", disable=["parser", "ner"])

    # 2. Stop words listelerini oluştur
    stop_en = set(stopwords.words("english"))
    stop_de = set(stopwords.words("german"))
    stop_tr = set(stopwords.words("turkish"))
    # TR için sorunlu kelimeleri ekliyoruz
    stop_tr.update({"bi", "bir", "mi", "mu", "bunu", "şunu", "ona", "bu", "şu", "o"})

    # 3. Veriyi yükle
    df = pd.read_json("../Datasets/tickets_combined.jsonl", lines=True)

    # 4. Temizleme Fonksiyonları
    def pre_clean(t):
        # Regex ile temel karakter temizliği
        t = t.lower().replace("\\n", " ")
        t = re.sub(r"<[^>]+>", " ", t)
        t = re.sub(r"[^\w\sçğıöşü]", " ", t)
        t = re.sub(r"\d+", " ", t)
        return re.sub(r"\s+", " ", t).strip()

    def clean_text_test(text, lang):
        t = pre_clean(text)

        if lang == "en":
            doc = nlp_en(t)
            tokens = [token.lemma_ for token in doc if token.text not in stop_en and len(token.text) > 1]
        elif lang == "de":
            doc = nlp_de(t)
            tokens = [token.lemma_ for token in doc if token.text not in stop_de and len(token.text) > 1]
        elif lang == "tr":
            doc = nlp_tr(t)
            tokens = [token.lemma_ for token in doc if token.text not in stop_tr and len(token.text) > 1]
        else:
            return t

        return " ".join(tokens)

    # 5. Test İşlemi (Her dilden 1 örnek)
    sample_en = df[df["language"]=="en"]["body"].iloc[0]
    sample_de = df[df["language"]=="de"]["body"].iloc[0]
    sample_tr = df[df["language"]=="tr"]["body"].iloc[0]

    print("=== TÜRKÇE (spaCy ile) ===")
    print("TR Orijinal:", sample_tr[:250])
    print("TR Temiz   :", clean_text_test(sample_tr, "tr")[:250])

    print("\n=== İNGİLİZCE ===")
    print("EN Orijinal:", sample_en[:250])
    print("EN Temiz   :", clean_text_test(sample_en, "en")[:250])

    print("\n=== ALMANCA ===")
    print("DE Orijinal:", sample_de[:250])
    print("DE Temiz   :", clean_text_test(sample_de, "de")[:250])

c:\Users\Mustafa\s_env\Lib\site-packages\spacy\util.py:918: UserWarning: [W094] Model 'tr_core_news_lg' (3.4.2) specifies an under-constrained spaCy version requirement: >=3.8.0. This can lead to compatibility problems with older versions, or as new spaCy versions are released, because the model may say it's compatible when it's not. Consider changing the "spacy_version" in your meta.json to a version range, with a lower and upper pin. For example: >=3.8.2,<3.9.0
  warnings.warn(warn_msg)


=== TÜRKÇE (spaCy ile) ===
TR Orijinal: <isim>,

Son iki haftadır Antalya şubesinde çalışırken faturamda aşırı miktarda bir fark çıkıyor ve bunu nasıl düzelteceğimi bilmiyorum. Lütfen acilen bu durumu inceleyin çünkü kredi kartımın son geçerlilik tarihine çok yaklaşıyoruz.

Çok endişeliyim
TR Temiz   : son iki hafta antalya şube çalış faturamda aşırı miktar fark çık düzelteceğimi bilmiyorum lütfen ac durum incele kredi kartımın son geçerli tarih yaklaşa endişeli lutfen kısa süre dön yapmanızı rica et

=== İNGİLİZCE ===
EN Orijinal: Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenienc
EN Temiz   : dear customer support team write report significant problem centralized account management portal currently appear offline outage block access account setting lead substantial inconvenience attempt 

In [2]:
from zemberek import (
    TurkishMorphology,
    TurkishSentenceNormalizer,
    TurkishSpellChecker,
)

morphology = TurkishMorphology.create_with_defaults()

2026-07-07 12:46:03,318 - zemberek.morphology.turkish_morphology - INFO
Msg: TurkishMorphology instance initialized in 6.438724517822266



In [ ]:
import re

from matplotlib.pyplot import text

def clean_text(text, language, negation_marker="_NEG"):
    """Lowercasing + placeholder/HTML/noktalama/sayi temizligi + stop-word + lemmatization.
    TR icin: sentence-level disambiguation + negation bilgisini lemma'ya ekleme.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    text = text.lower()
    text = text.replace("\\n", " ")
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^\w\sçğıöşü]", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return ""

    if language == "en":
        doc = nlp_en(text)
        tokens = [t.lemma_ for t in doc if t.text not in stop_en and len(t.text) > 1]

    elif language == "de":
        doc = nlp_de(text)
        tokens = [t.lemma_ for t in doc if t.text not in stop_de and len(t.text) > 1]

    elif language == "tr":
        sentence_analysis = morphology.analyze_sentence(text)
        disambiguated = morphology.disambiguate(text, sentence_analysis)
        best_analyses = disambiguated.best_analysis()

        tokens = []
        for word_analysis in best_analyses:
            if not hasattr(word_analysis, "surface_form"):
                continue

            surface_text = word_analysis.surface_form().lower()

            if surface_text in stop_tr or len(surface_text) <= 1:
                continue

            lemma = str(word_analysis.item.lemma).lower()
            if lemma == "unk":
                lemma = surface_text

            # Negation kontrolu: format_string icinde ":Neg" morfemi var mi
            fstring = word_analysis.format_string() if hasattr(word_analysis, "format_string") else ""
            is_negative = bool(re.search(r":Neg\b", fstring))

            if is_negative:
                tokens.append(lemma + negation_marker)
            else:
                tokens.append(lemma)

    else:
        tokens = text.split()

    return " ".join(tokens)


# --- TEST ---
sample_en = df[df["language"]=="en"]["body"].iloc[0]
sample_de = df[df["language"]=="de"]["body"].iloc[0]
sample_tr = df[df["language"]=="tr"]["body"].iloc[0]

print("EN temiz:", clean_text(sample_en, "en")[:150])
print("DE temiz:", clean_text(sample_de, "de")[:150])
print()
print("TR orijinal:", sample_tr[:250])
print("TR temiz   :", clean_text(sample_tr, "tr")[:250])

EN temiz: dear customer support team write report significant problem centralized account management portal currently appear offline outage block access account
DE temiz: geehrt Support Team möchten gravierend Sicherheitsvorfall melden gegenwärtig mehrere Komponent unser Infrastruktur betreffen betroffen Gerät umfassen 

TR orijinal: <isim>,

Son iki haftadır Antalya şubesinde çalışırken faturamda aşırı miktarda bir fark çıkıyor ve bunu nasıl düzelteceğimi bilmiyorum. Lütfen acilen bu durumu inceleyin çünkü kredi kartımın son geçerlilik tarihine çok yaklaşıyoruz.

Çok endişeliyim
TR temiz   : son iki hafta antalya şube çalışmak fatura aşırı miktar fark çıkmak düzelmek bilmek lütfen acilen durum incelemek kredi kart son geçerli tarih yaklaşmak endişe lutfen kısa süre dönmek yapmak rica etmek
